In [1]:
# import required python modules
import os
import fitz
import pandas as pd

In [1]:
# set input and output paths
base_dir = r"D:\Egypt-Opportunity-Radar-Task3\data\raw\production"
in_path = os.path.join(base_dir, "task3_imc_investment_guide.pdf")
out_dir = os.path.join(base_dir, "cleaned")
os.makedirs(out_dir, exist_ok=True)

In [1]:
# open raw guide pdf
doc = fitz.open(in_path)
rows = []

In [1]:
# extract product fact sheets
for idx, page in enumerate(doc):
    lines = [ln.strip() for ln in page.get_text("text").splitlines() if ln.strip()]
    imgs = len(page.get_images())
    printed_page = ""
    product = ""
    if "Investment Opportunity" in lines and "Fact Sheet" in lines:
        p_idx = lines.index("Fact Sheet")
        if p_idx > 1:
            printed_page = lines[p_idx - 2]
        if p_idx + 1 < len(lines):
            product = lines[p_idx + 1]
    rows.append({
        "pdf_page": idx + 1,
        "printed_page": printed_page,
        "product_title": product,
        "image_count": imgs,
        "line_count": len(lines),
        "cleaned_text": " ".join(lines)
    })

In [1]:
# convert to cleaned dataframe
df_guide = pd.DataFrame(rows)

In [1]:
# save cleaned guide outputs
csv_out = os.path.join(out_dir, "cleaned_task3_imc_investment_guide.csv")
df_guide.to_csv(csv_out, index=False, encoding="utf-8-sig")

In [1]:
xlsx_out = os.path.join(out_dir, "cleaned_task3_imc_investment_guide.xlsx")
with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
    df_guide[df_guide["product_title"] != ""].to_excel(writer, sheet_name="product_fact_sheets", index=False)
    df_guide.to_excel(writer, sheet_name="all_pages", index=False)